In a nutshell: Imagine you are a food brand who want to enter their ingredient list - except it's a form that asks select source, process type, function etc and it doesn't let you enter any string you want - you must select from the list that's given.

What we are creating is just that, and to create it - we need to know what options we show under each category. Below is a sample case study for foritifcation session along with sweeteners overview - what you will be working on as the first rabbit hole.

Think from a brand's pov, they used some sort of sweetener - when they search for it from the list - what will their intuitive search be?

Now this can be answered in 2 ways,

1. Checking what's already on the back of packet labels to see what kind of ingredients they use for sweetening
2. Checking what the best practises in industry are and how the industry, law and basically what's on textbook as a sweetener, think of the difference as this one being what's written down and the 1st being what was found in the wild.

Track 1 handles the first question and finds what can't be answered from in the wild data and sends the question over to Track 2

Please know it's normal to dream of sugar canes when you do this, we got you <33

# From label rows to taxonomy: how a category session works

This is the mental process, not the file mechanics. File mechanics (which file is active, when to commit, how cp/ works) are in the `isrl-file-lifecycle` skill. This notebook is about how to think through a category session — what question you are answering, how you make decisions, and what the output is supposed to do.

Run the cells in order. Each code cell loads real data from the working files. The tables are not mock examples — they are actual rows and taxonomy entries produced by the fortification and sweetener sessions.

## The governing question

The IFID schema is a form. A brand fills it in when they register a product. Every category session is answering one question:

> If a brand fills in this form and picks *this ingredient type*, what fields appear — and what do those fields show as options?

That is the governing question. Every decision in a session is tested against it.

Not: *what is the correct scientific classification of this substance?*  
Not: *what does the label say?*  
Not: *is this a real ingredient?*

Those are inputs. The governing question is always about what the form needs to do.

When a decision is hard — when two reasonable people would disagree about where a row belongs — the way to break the tie is to ask what the form field looks like in each case. If one answer produces a form field that makes sense for a brand user, and the other doesn't, you have your answer.

## Case study: fortification

### What we started with

Before any reasoning starts, a session begins with raw rows. These are actual rows from the fortification session — the variant strings, sources, what the label said (`f_explicit`), nutrient class assignment, and zone score. Run the cell to see what the raw input looks like.

In [1]:
import pandas as pd

df = pd.read_csv('fortification_agent/fortification_agent_working.csv')
display_cols = ['variant', 'source', 'f_explicit', 'f_could_be', 'nutrient_class', 'zone']
df[display_cols].head(12)

,variant,source,f_explicit,f_could_be,nutrient_class,zone
0,l-hpc,synthetic,base ingredient,NaN,amino_acid,2
1,biotin,synthetic,base ingredient,NaN,vitamin_water_soluble,2
2,iodate,mineral,base ingredient,NaN,mineral_nutrient,2
3,niacin,synthetic,base ingredient,NaN,vitamin_water_soluble,2
4,valine,synthetic,base ingredient,NaN,amino_acid,2
5,lysine,synthetic,base ingredient,NaN,amino_acid,2
6,taurine,synthetic,NaN,NaN,amino_acid,1
7,thiamin,synthetic,NaN,NaN,vitamin_water_soluble,1
8,leucine,synthetic,NaN,NaN,amino_acid,1
9,d-biotin,synthetic,NaN,NaN,vitamin_water_soluble,1


### The reasoning: agent vs. carrier, boundary decisions

**The first non-obvious thing**

*Fortification agent* means what was **added**, not what it was added to. Riboflavin is in the taxonomy. Wheat flour — the thing riboflavin was added to — is not. The taxonomy is a catalogue of **agents**, not carriers.

This sounds obvious once stated, but it is easy to get wrong. If you scan a label that says "fortified wheat flour thiamin", you might think "this is about wheat flour" and try to build a wheat flour entry. But wheat flour is a base ingredient. The fortification taxonomy is about thiamin — the thing added to it.

**Why that distinction matters for the form**

IFID's schema puts the checkbox on the **ingredient entry** (edible oil, wheat flour, salt). When a brand selects an ingredient and checks "is this fortified?", the next field asks "which agents?". The taxonomy populates that dropdown.

If you build the taxonomy with carriers in it (wheat flour, edible oil), the dropdown becomes meaningless. A brand would open "fortification agents" and see wheat flour next to riboflavin. The form breaks.

The agent-vs-carrier distinction isn't a conceptual nicety. It determines whether the form field is usable.

**The boundary decision: what crossed the line into health_supplement**

l-carnitine, creatine, and whey peptides appeared in the rows with the word "fortified" on the label. But they are regulated under FSSAI's health supplement rules, not under food fortification standards. The FSSAI fortification mandate covers: edible oil (A, D), wheat flour (iron, folic acid, B12, zinc), rice, milk, double-fortified salt.

l-carnitine is not on that list. Neither is creatine. The word "fortified" on the label was being used loosely — the manufacturer meant "enhanced". We drew the line by asking: what would FSSAI's fortification mandate cover? Everything outside that line went to `health_supplement`.

**The compound declaration problem**

"Fortified wheat flour thiamin" bundles carrier + agent in one string. The label format does not know about our schema — that compound string is a limit of what current label conventions express, not a data error. These rows were flagged `flag_compound_product`.

**What the session produced**

53 canonical fortification agents. The form design questions were answered: yes to the "is this fortified?" checkbox; the dropdown shows agents filtered by nutrient class relevant to each carrier.

In [2]:
# Rows reclassified to health_supplement during the fortification session
hs = pd.read_csv('health_supplement/pending_enrichment.csv')
display_cols = ['variant', 'source', 'f_explicit', 'f_could_be', 'nutrient_class']

# Rows with nutrient_class filled in went through fortification scoring
hs_scored = hs[hs['nutrient_class'].notna()]
print(f'Total in health_supplement queue: {len(hs)} rows')
print(f'Rows from fortification session (nutrient_class assigned): {len(hs_scored)}')
hs_scored[display_cols]

Total in health_supplement queue: 29 rows
Rows from fortification session (nutrient_class assigned): 6


,variant,source,f_explicit,f_could_be,nutrient_class
15,l-camitine,synthetic,base ingredient,NaN,amino_acid
16,l-carnitine,synthetic,NaN,NaN,amino_acid
17,whey peptides,milk,base ingredient,NaN,amino_acid
18,glutamine peptides,synthetic,base ingredient,NaN,amino_acid
26,creatine monohydrate micronized,synthetic,base ingredient,NaN,amino_acid
28,calcium b-hydroxy-b-methylbutyrate monohydrate...,synthetic,base ingredient,NaN,mineral_nutrient


In [3]:
# The compound declaration rows — cannot be cleanly assigned to any category
wdf = pd.read_csv('core/all_variants_working.csv')
compound = wdf[wdf['f_revised'] == 'flag_compound_product']
print(f'Compound declaration rows: {len(compound)}')
compound[['variant', 'source', 'f_explicit', 'f_could_be']]

Compound declaration rows: 3


,variant,source,f_explicit,f_could_be
1193,butterscotch pieces,milk | sugarcane,sweetener|flavouring agent,NaN
1552,sweetened condensed skimmed milk,milk,sweetener|base ingredient,NaN
1631,sweetened condensed partly skimmed milk,milk,base ingredient|sweetener,NaN


### What the session produced: taxonomy output

The session produced 53 canonical entries. Here is what a taxonomy entry looks like — this is the data that populates the form dropdown. Each row is one entry a brand could select when declaring a fortification agent.

In [4]:
import json

with open('fortification_agent/fortification_taxonomy.json') as f:
    tax = json.load(f)

entries = list(tax['fortification_agents'].values())
print(f'Total taxonomy entries: {len(entries)}')

tax_df = pd.DataFrame(entries)
tax_df[['canonical_name', 'nutrient_type', 'regulatory_category', 'source_type', 'is_group_declaration']].head(8)

Total taxonomy entries: 53


,canonical_name,nutrient_type,regulatory_category,source_type,is_group_declaration
0,Amino Acids,group_declaration,unregulated,UNSURE,True
1,Arginine,amino_acid,fssai_hsnfsdu,microbial,False
2,Biotin,vitamin_b_group,fssai_hsnfsdu,synthetic,False
3,Calcium Salt,mineral_salt,unregulated,mineral,True
4,Cholecalciferol,vitamin_fat_soluble,fssai_mandatory_fortification,synthetic,False
5,Cyanocobalamin,vitamin_b_group,fssai_mandatory_fortification,synthetic,False
6,DL-Methionine,amino_acid,fssai_hsnfsdu,synthetic,False
7,Electrolytes,group_declaration,unregulated,UNSURE,True


## The mechanics: from rows to taxonomy

### Multiple label variants → one canonical entry

"zinc sulphate" and "zinc sulphate monohydrate" appear as separate rows. They refer to the same substance. The taxonomy carries one entry — **Zinc Sulphate** — with the variants collapsed into it.

| Field | Value |
|---|---|
| `canonical_name` | Zinc Sulphate |
| `nutrient_type` | mineral |
| `regulatory_category` | fssai_hsnfsdu |
| `source_type` | synthetic |
| `is_group_declaration` | False |
| `source_row_variants` | zinc sulphate, zinc sulphate monohydrate |

The `is_group_declaration` flag is False — this is a precise substance. Compare with "Amino Acids" in the taxonomy above (`is_group_declaration: True`) — that entry represents a real label string that cannot be resolved to a single substance. It stays in the taxonomy for matchability, but the flag tells downstream systems not to route it to precise INS lookups.

### OCR noise is corrected, not stripped

"phenylalaline" is an OCR error for "phenylalanine". We correct it in the taxonomy but keep the original string in `source_row_variants` so the bad string is still traceable. If a future label produces the same OCR error, the system can still match it.

### `source_type` is not academic metadata

It determines dietary flags. If an entry is `source_type: animal`, that fires halal and vegan checks in the downstream system. A brand declaring an ingredient with the wrong source_type will receive incorrect dietary flags. Getting source_type wrong is a data quality error with downstream consequences.

## Non-obvious things a newcomer needs to know

**1. `f_revised` is a working classification, not a final IFID field.**  
It is our best-guess routing of a label string to an analysis lane. When a session closes, the taxonomy entries — not `f_revised` — are what the IFID schema uses. `f_revised` exists to organise the work, not to be published.

**2. The category name on the label is not always right.**  
Brands use "sweetener" to mean artificial sweeteners, table sugar, honey, and dates — sometimes in the same product declaration. Brands use "fortified" to mean FSSAI-mandated vitamin A and also creatine monohydrate. The session draws the line the label didn't. That is the job.

**3. INS number ≠ single substance.**  
INS 960 (Steviol glycosides) covers multiple compounds: Reb-A, Reb-M, stevioside, and others. INS 471 covers a class with plant and animal sources — the source_type is not derivable from the INS number alone.

**4. The taxonomy is a lookup table for Indian F&B labels, not a global ontology.**  
Gaps in the taxonomy mean something didn't appear in our data — not that it doesn't exist. When a substance is absent, check whether it belongs in scope, then add it if it does.

**5. Every sentence in a Track A report is one of four things.**  
- A verified fact from the data  
- An inference from that fact (label it as such)  
- An open question the data raised but cannot answer  
- A design question for the schema  

Mixing these up — presenting inferences as facts, or design questions as findings — is where revision cycles come from.

## Applying this to the sweetener session

When a brand searches for "sweetener" in the IFID form, they will find ingredients from three distinct classes — all legitimately called sweeteners, all appearing in our label data, all needing the `sweetener` functional tag so they surface in search. But the form fields they require are different:

| Class | What it is | Fields the form needs |
|---|---|---|
| **A — INS additive sweeteners** | Regulated food additives with assigned INS numbers: aspartame, acesulfame K, sucralose, steviol glycosides, isomalt, xylitol | INS number, source_type, dietary flags |
| **B — Processed sugar ingredients** | Refined/derived sugar products: sucrose, glucose, fructose, corn syrup, invert sugar, glucose syrup | Source (e.g. sugarcane), processing method (EMF process types), sugar_form |
| **C — Natural sweetening materials** | Agricultural raw materials used as sweeteners: honey, jaggery, molasses, palm sugar, dates, raisins | Fields to be established based on what these entries require |

Each class has different field requirements. They cannot share a single form field structure — a brand declaring aspartame (Class A) needs an INS lookup; a brand declaring jaggery (Class C) does not. The `sweetener` tag is what makes all three discoverable from one search; the class is what determines which fields appear after selection.

**The core task for the sweetener session:**

From the 90 rows currently in the sweetener lane, identify which specific ingredients from our label data belong to Class A, Class B, and Class C. That inventory is the output. What the IFID form does with each class — what fields it shows, what Track B needs to verify — follows from that enumeration.

In [5]:
wdf = pd.read_csv('core/all_variants_working.csv')
sw_all = wdf[wdf['f_revised'].str.contains('sweetener', na=False)]

print(f'Total rows with sweetener in f_revised: {len(sw_all)}')
print()
print('--- f_revised breakdown ---')
print(sw_all['f_revised'].value_counts().to_string())
print()

# Sample rows from each recognisable subtype
# INS additive sweeteners — have ins_number or are known additives
ins_sw = sw_all[sw_all['ins_number'].notna()]
print(f'--- Rows with INS number assigned (Class A candidates): {len(ins_sw)} ---')
print(ins_sw[['variant', 'ins_number', 'source', 'f_revised']].to_string(index=False))
print()

# INS check still needed
ins_check = wdf[wdf['f_revised'] == 'sweetener|bulking agent|ins_check_needed']
print(f'--- INS check needed (Class A candidates, INS not yet confirmed): {len(ins_check)} ---')
print(ins_check[['variant', 'source', 'ins_number', 'f_revised']].to_string(index=False))
print()

# Pre-session reclassifications — rows that left the sweetener lane entirely
print('--- Pre-session reclassifications ---')
print('colour|additives (caramel powder):')
cp = wdf[wdf['f_revised'] == 'colour|additives']
print(cp[['variant', 'source', 'f_explicit', 'f_revised']].to_string(index=False))
print()
print('flag_compound_product (butterscotch, condensed milk):')
comp = wdf[wdf['f_revised'] == 'flag_compound_product']
print(comp[['variant', 'source', 'f_explicit', 'f_revised']].to_string(index=False))
print()
print('health_supplement (FOS/gudmar family — sample):')
fos = wdf[
    (wdf['f_revised'] == 'health_supplement') &
    (wdf['variant'].str.contains('fructo|fos|gudmar', case=False, na=False))
]
print(fos[['variant', 'source', 'f_revised']].to_string(index=False))

Total rows with sweetener in f_revised: 90

--- f_revised breakdown ---
f_revised
sweetener                                   88
sweetener|bulking agent|ins_check_needed     2

--- Rows with INS number assigned (Class A candidates): 0 ---
Empty DataFrame
Columns: [variant, ins_number, source, f_revised]
Index: []

--- INS check needed (Class A candidates, INS not yet confirmed): 2 ---
            variant         source  ins_number                                f_revised
  corn syrup solids           corn         NaN sweetener|bulking agent|ins_check_needed
dried glucose syrup corn | tapioca         NaN sweetener|bulking agent|ins_check_needed

--- Pre-session reclassifications ---
colour|additives (caramel powder):
       variant    source       f_explicit        f_revised
caramel powder sugarcane colour|sweetener colour|additives

flag_compound_product (butterscotch, condensed milk):
                                variant           source                 f_explicit             f_rev